# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Memes820/Flyrank-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [20]:
!git clone https;//github.com/Memes820/Flyrank-Internship.git
%cd Flyrank-Internship

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf ( TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
print("Connected! Ready to query the warehouse.")

fatal: repository 'https' does not exist
/bin/bash: line 1: //github.com/Memes820/Flyrank-Internship.git: No such file or directory
[Errno 2] No such file or directory: 'Flyrank-Internship'
/content
Connected! Ready to query the warehouse.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
 One row = one content page, for one client, on one day(client_hash_id + content_hash_id + report_date).I will use a mid panel month = 2026-03, to avoid the sealed final month and rate limits.

In [21]:
query = f"""
SELECT COUNT(*) as row_count
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
"""
con.sql(query).show()

┌───────────┐
│ row_count │
│   int64   │
├───────────┤
│   9841378 │
└───────────┘



## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
Features (knowable before the decision): impressions,clicks,avg_position,ctr,sessions,content_age_days.
Label/proxy:whether a pages impressions declined from the prior month (I will compute this later, not stored directly).
COntext:client_hash_id, content_id used for joins/grouping only, not as predictive features. Excluded:any FlyRank product decison flag (health_score, priority_score,action_type), these are not in this dataset, and even if they were, using them would just teach the model to copy an existing rule instead of finding real signal.

In [22]:
query = f"""
SELECT*
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
LIMIT 5
"""
con.sql(query).show()

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬───────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_clau

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*
Query 1 confirms the grain: one row per client=content=day, no duplicates.
Query 2 shows the row count and date span for this months slice
Query 3 checks availability of how many rows have real impressions data, filtered with IS TRUE where relevant.

In [23]:
# Query 1: grain check- no duplicate rows per key
q1 = f"""
SELECT client_hash_id, content_hash_id,report_date, COUNT(*) as n
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
GROUP BY 1,2,3
HAVING COUNT(*) > 1
LIMIT 5
"""
print("Duplicate grain rows (should be empty):")
con.sql(q1).show()

#Quert 2: row count + date span
q2 = f"""
SELECT COUNT(*) as rows, MIN(report_date) as min_date, MAX(report_date) as max_date
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
"""
print("Row count and date span:")
con.sql(q2).show()

#Query 3: availability check
q3 = f"""
SELECT COUNT(*) as rows_with_impressions
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE gsc_impressions IS NOT NULL
"""
print("Rows with real impressions data:")
con.sql(q3).show()

Duplicate grain rows (should be empty):


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬─────────────────┬─────────────┬───────┐
│ client_hash_id │ content_hash_id │ report_date │   n   │
│    varchar     │     varchar     │    date     │ int64 │
├────────────────┴─────────────────┴─────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘

Row count and date span:
┌─────────┬────────────┬────────────┐
│  rows   │  min_date  │  max_date  │
│  int64  │    date    │    date    │
├─────────┼────────────┼────────────┤
│ 9841378 │ 2026-03-01 │ 2026-03-31 │
└─────────┴────────────┴────────────┘

Rows with real impressions data:
┌───────────────────────┐
│ rows_with_impressions │
│         int64         │
├───────────────────────┤
│               9841378 │
└───────────────────────┘



## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*
This slice can not tell me about clients whose tracking started after March 2026 or about GA4-only metrics for clients who only have search data early on It also cannot prove
causation only association. Since Im using one mid panel month, seasonal effects specific to other months are not captured here either

In [24]:
# No additional query needed - this section is reflection only.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.